In [ ]:
import json

DATASET_PATH = '/home/vit/Projects/cryptobench/osf-setup/cryptobench/cryptobench-dataset/dataset.json'
with open(DATASET_PATH, 'r') as file:
    data = json.load(file)

main_structures = {}
for apo_id, holo_structures in data.items():

    # only for the purpose of sanity check
    is_multichain = False

    # find main holo structure
    for structure in holo_structures:
        if structure['is_main_holo_structure']:
            if '-' in structure['apo_chain'] or '-' in structure['holo_chain']:
                # this is a multichain structure, skip
                is_multichain = True
                break
            main_structures[apo_id] = structure
            break

    assert apo_id in main_structures or is_multichain, f"Main holo structure not found for {apo_id}"

    # remove redundant fields
    if not is_multichain:
        del main_structures[apo_id]['is_main_holo_structure']

In [8]:
main_structures['1a4u']

{'uniprot_id': 'P10807',
 'holo_pdb_id': '3rj9',
 'holo_chain': 'C',
 'apo_chain': 'B',
 'ligand': 'NAD',
 'ligand_index': '850',
 'ligand_chain': 'C',
 'apo_pocket_selection': ['B_12',
  'B_14',
  'B_15',
  'B_16',
  'B_17',
  'B_18',
  'B_37',
  'B_38',
  'B_62',
  'B_63',
  'B_64',
  'B_65',
  'B_91',
  'B_92',
  'B_93',
  'B_94',
  'B_106',
  'B_136',
  'B_137',
  'B_138',
  'B_151',
  'B_155',
  'B_181',
  'B_182',
  'B_183',
  'B_184',
  'B_186',
  'B_187',
  'B_188',
  'B_189'],
 'holo_pocket_selection': ['C_12',
  'C_14',
  'C_15',
  'C_16',
  'C_17',
  'C_18',
  'C_37',
  'C_38',
  'C_62',
  'C_63',
  'C_64',
  'C_65',
  'C_91',
  'C_92',
  'C_93',
  'C_94',
  'C_106',
  'C_136',
  'C_137',
  'C_138',
  'C_151',
  'C_155',
  'C_181',
  'C_182',
  'C_183',
  'C_184',
  'C_186',
  'C_187',
  'C_188',
  'C_189'],
 'apo_pymol_selection': '1a4u and ( (chain B and resi 12+14+15+16+17+18+37+38+62+63+64+65+91+92+93+94+106+136+137+138+151+155+181+182+183+184+186+187+188+189) )',
 'holo

In [ ]:
PATH = '/home/vit/Projects/cryptobench/src/other/cryptobench-main-structures-without-SASA-and-volume.json'
with open(PATH, 'w') as outfile:
    json.dump(main_structures, outfile, indent=4)

## Get SASA & Volume

In [24]:
import numpy as np
import pandas as pd
import json
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
INPUT_PATH = '/home/vit/Projects/cryptobench/src/other/cryptobench-main-structures-without-SASA-and-volume.json'
PREFILTERED_CSV = '/home/vit/Projects/cryptobench/data/B-create-dataset/ahoj-v2/dataset_dataframe.csv'

with open(INPUT_PATH) as f:
    cryptic_dataset = json.load(f)
dataset_base_pd = pd.read_csv(PREFILTERED_CSV)


for apo_pdb_id, holo_structure in cryptic_dataset.items():
    holo_pdb_id = holo_structure['holo_pdb_id']
    holo_chain_id = holo_structure['holo_chain']
    apo_chain_id = holo_structure['apo_chain']
    ligand = holo_structure['ligand']
    ligand_index = int(holo_structure['ligand_index'])
    ligand_chain = holo_structure['ligand_chain']
    record = dataset_base_pd[
        (dataset_base_pd['apo_structure'] == apo_pdb_id) &
        (dataset_base_pd['holo_structure'] == holo_pdb_id) &
        (dataset_base_pd['apo_chains3'] == apo_chain_id) &
        (dataset_base_pd['holo_chains3'] == holo_chain_id) &
        (dataset_base_pd['ligand'] == ligand) &
        (dataset_base_pd['ligand_index'] == ligand_index) &
        (dataset_base_pd['ligand_chain'] == ligand_chain)
        ]
    
    assert len(record) == 1
    apo_sasa = record['apo_sasa'].iloc[0]       
    holo_sasa = record['holo_sasa'].iloc[0]       
    apo_volume = record['apo_volume'].iloc[0]       
    holo_volume = record['holo_volume'].iloc[0]       
    
    if np.isnan(apo_volume):
        apo_volume = 'NaN'
    if np.isnan(holo_volume):
        holo_volume = 'NaN'

    cryptic_dataset[apo_pdb_id]['apo_sasa'] = apo_sasa
    cryptic_dataset[apo_pdb_id]['holo_sasa'] = holo_sasa
    cryptic_dataset[apo_pdb_id]['apo_volume'] = apo_volume
    cryptic_dataset[apo_pdb_id]['holo_volume'] = holo_volume


/tmp/ipykernel_7155/2158605779.py:11: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset_base_pd = pd.read_csv(PREFILTERED_CSV)


In [25]:
PATH = '/home/vit/Projects/cryptobench/src/other/cryptobench-main-structures.json'
with open(PATH, 'w') as outfile:
    json.dump(cryptic_dataset, outfile, indent=4)